# VetCare Herts: synthetic veterinary clinic database

This notebook generates every row of `vetcare_clinics.db`, an SQLite database for a fictional group of
seven veterinary practices in Hertfordshire and Bedfordshire. No data was downloaded: all values are
produced here with NumPy random distributions and the Faker library (UK locale), using a fixed seed so
the database can be rebuilt exactly.

**Sections**
1. Setup and seed
2. Schema (DDL with all constraints)
3. Reference lists
4. Table generators (in foreign-key order)
5. Injecting deliberate data-quality issues (missing values and duplicates)
6. Loading into SQLite with foreign keys enforced
7. Validation checks
8. Example queries

In [1]:
try:
    import faker
except ImportError:          # Colab and fresh environments
    %pip install -q faker

In [2]:
import sqlite3, os, random
from datetime import date, datetime, timedelta
import numpy as np
import pandas as pd
from faker import Faker

SEED = 2026
rng = np.random.default_rng(SEED)
random.seed(SEED)
fake = Faker('en_GB')
Faker.seed(SEED)

DB_PATH = 'vetcare_clinics.db'
START, END = date(2024, 1, 1), date(2026, 9, 30)   # appointment window

## 2. Schema

In [3]:
SCHEMA = """
PRAGMA foreign_keys = ON;

CREATE TABLE clinics (
    clinic_id          INTEGER PRIMARY KEY,
    clinic_name        TEXT    NOT NULL UNIQUE,
    town               TEXT    NOT NULL,
    county             TEXT    NOT NULL CHECK (county IN ('Hertfordshire','Bedfordshire')),
    postcode_district  TEXT    NOT NULL CHECK (length(postcode_district) BETWEEN 3 AND 4 AND substr(postcode_district, 1, 2) GLOB '[A-Z][A-Z]'),
    opened_year        INTEGER NOT NULL CHECK (opened_year BETWEEN 1955 AND 2026),
    consult_rooms      INTEGER NOT NULL CHECK (consult_rooms BETWEEN 1 AND 12),
    has_24h_emergency  INTEGER NOT NULL DEFAULT 0 CHECK (has_24h_emergency IN (0,1))
);

CREATE TABLE vets (
    vet_id          INTEGER PRIMARY KEY,
    home_clinic_id  INTEGER NOT NULL REFERENCES clinics(clinic_id) ON UPDATE CASCADE ON DELETE RESTRICT,
    first_name      TEXT    NOT NULL CHECK (length(trim(first_name)) BETWEEN 1 AND 45),
    last_name       TEXT    NOT NULL CHECK (length(trim(last_name)) BETWEEN 1 AND 45),
    rcvs_number     TEXT    NOT NULL UNIQUE CHECK (rcvs_number GLOB '[0-9][0-9][0-9][0-9][0-9][0-9][0-9]'),
    grade           TEXT    NOT NULL CHECK (grade IN ('Graduate','Vet','Senior Vet','Clinical Director')),
    qualified_year  INTEGER NOT NULL CHECK (qualified_year BETWEEN 1980 AND 2026),
    work_email      TEXT    NOT NULL UNIQUE CHECK (work_email LIKE '%_@vetcareherts.co.uk')
);

CREATE TABLE vet_rota (
    vet_id     INTEGER NOT NULL REFERENCES vets(vet_id) ON DELETE CASCADE,
    clinic_id  INTEGER NOT NULL REFERENCES clinics(clinic_id) ON DELETE CASCADE,
    weekday    INTEGER NOT NULL CHECK (weekday BETWEEN 1 AND 7),       -- 1 = Monday
    shift      TEXT    NOT NULL CHECK (shift IN ('Full day','Morning','Afternoon')),
    PRIMARY KEY (vet_id, clinic_id, weekday)
);

CREATE TABLE owners (
    owner_id           INTEGER PRIMARY KEY,
    first_name         TEXT    NOT NULL CHECK (length(trim(first_name)) BETWEEN 1 AND 45),
    last_name          TEXT    NOT NULL CHECK (length(trim(last_name)) BETWEEN 1 AND 45),
    email              TEXT    CHECK (email IS NULL OR (instr(email, '@') > 1 AND email LIKE '%@%.%')),
    mobile             TEXT    CHECK (mobile IS NULL OR (length(mobile) = 12 AND substr(mobile, 1, 2) = '07' AND mobile NOT GLOB '*[^0-9 ]*')),
    postcode_district  TEXT    NOT NULL CHECK (length(postcode_district) BETWEEN 3 AND 4 AND substr(postcode_district, 1, 2) GLOB '[A-Z][A-Z]'),
    home_clinic_id     INTEGER NOT NULL REFERENCES clinics(clinic_id) ON DELETE RESTRICT,
    registered_on      TEXT    NOT NULL CHECK (date(registered_on) = registered_on),
    marketing_consent  INTEGER NOT NULL DEFAULT 0 CHECK (marketing_consent IN (0,1))
);

CREATE TABLE pets (
    pet_id         INTEGER PRIMARY KEY,
    owner_id       INTEGER NOT NULL REFERENCES owners(owner_id) ON DELETE CASCADE,
    pet_name       TEXT    NOT NULL CHECK (length(pet_name) BETWEEN 1 AND 30),
    species        TEXT    NOT NULL CHECK (species IN ('Dog','Cat','Rabbit','Guinea pig','Bird','Reptile')),
    breed          TEXT    NOT NULL,
    sex            TEXT    NOT NULL CHECK (sex IN ('Male','Female','Unknown')),
    date_of_birth  TEXT    NOT NULL CHECK (date(date_of_birth) = date_of_birth),
    weight_kg      REAL    CHECK (weight_kg IS NULL OR weight_kg BETWEEN 0.01 AND 110),
    neutered       INTEGER NOT NULL DEFAULT 0 CHECK (neutered IN (0,1)),
    microchip_no   TEXT    UNIQUE CHECK (microchip_no IS NULL OR (length(microchip_no) = 15 AND microchip_no NOT GLOB '*[^0-9]*')),
    UNIQUE (owner_id, pet_name, date_of_birth)
);

CREATE TABLE treatments (
    treatment_code  TEXT PRIMARY KEY CHECK (treatment_code GLOB '[A-Z][A-Z][A-Z]-[0-9][0-9][0-9]'),
    description     TEXT NOT NULL UNIQUE,
    category        TEXT NOT NULL CHECK (category IN ('Consultation','Vaccination','Diagnostics','Surgery',
                                                      'Dental','Medication','Parasite control')),
    list_price_gbp  REAL NOT NULL CHECK (list_price_gbp >= 0),
    requires_sedation INTEGER NOT NULL DEFAULT 0 CHECK (requires_sedation IN (0,1))
);

CREATE TABLE appointments (
    appointment_id  INTEGER PRIMARY KEY,
    pet_id          INTEGER NOT NULL REFERENCES pets(pet_id) ON DELETE CASCADE,
    vet_id          INTEGER NOT NULL REFERENCES vets(vet_id) ON DELETE RESTRICT,
    clinic_id       INTEGER NOT NULL REFERENCES clinics(clinic_id) ON DELETE RESTRICT,
    start_time      TEXT    NOT NULL CHECK (datetime(start_time) = start_time),
    reason          TEXT    NOT NULL CHECK (reason IN ('Routine check','Vaccination','Illness','Injury',
                                                       'Dental','Surgery','Follow-up','Emergency')),
    triage_level    INTEGER NOT NULL CHECK (triage_level BETWEEN 1 AND 5),   -- 1 routine ... 5 critical
    status          TEXT    NOT NULL DEFAULT 'Completed' CHECK (status IN ('Completed','Cancelled','No-show')),
    body_temp_c     REAL    CHECK (body_temp_c IS NULL OR body_temp_c BETWEEN 35.0 AND 43.5),
    duration_min    INTEGER CHECK (duration_min IS NULL OR duration_min BETWEEN 5 AND 480),
    fee_gbp         REAL    NOT NULL DEFAULT 0 CHECK (fee_gbp >= 0),
    payment_method  TEXT    CHECK (payment_method IS NULL OR payment_method IN
                                   ('Card','Cash','Insurance claim','Pet health plan','Bank transfer'))
);

CREATE TABLE appointment_treatments (
    appointment_id  INTEGER NOT NULL REFERENCES appointments(appointment_id) ON UPDATE CASCADE ON DELETE CASCADE,
    treatment_code  TEXT    NOT NULL REFERENCES treatments(treatment_code) ON UPDATE CASCADE ON DELETE RESTRICT,
    quantity        INTEGER NOT NULL DEFAULT 1 CHECK (quantity BETWEEN 1 AND 20),
    line_price_gbp  REAL    NOT NULL CHECK (line_price_gbp >= 0),
    PRIMARY KEY (appointment_id, treatment_code)
);

CREATE INDEX idx_appt_pet   ON appointments(pet_id);
CREATE INDEX idx_appt_vet   ON appointments(vet_id, start_time);
CREATE INDEX idx_pets_owner ON pets(owner_id);
"""

## 3. Reference lists

In [4]:
CLINICS = [  # name, town, county, district, opened, rooms, 24h
    ('Ashbrook Veterinary Centre', 'St Albans', 'Hertfordshire', 'AL1', 1987, 6, 1),
    ('Comet Road Vets', 'Hatfield', 'Hertfordshire', 'AL10', 2009, 4, 0),
    ('Fairlands Animal Hospital', 'Stevenage', 'Hertfordshire', 'SG1', 1998, 8, 1),
    ('Cassiobury Pet Clinic', 'Watford', 'Hertfordshire', 'WD17', 2014, 5, 0),
    ('Wardown Veterinary Surgery', 'Luton', 'Bedfordshire', 'LU2', 2003, 5, 0),
    ('Gadebridge Vets', 'Hemel Hempstead', 'Hertfordshire', 'HP1', 2018, 3, 0),
    ('Parkway Veterinary Practice', 'Welwyn Garden City', 'Hertfordshire', 'AL8', 1976, 4, 0),
]
# nearby postcode districts that owners of each clinic live in
CATCHMENT = {1: ['AL1','AL2','AL3','AL4','AL5'], 2: ['AL9','AL10','AL4'], 3: ['SG1','SG2','SG3','SG4'],
             4: ['WD17','WD18','WD19','WD24','WD25'], 5: ['LU1','LU2','LU3','LU4'],
             6: ['HP1','HP2','HP3'], 7: ['AL7','AL8','AL6']}

# species: (share of pets, lifespan years, [(breed, mean adult kg, sd kg, share)])
SPECIES = {
    'Dog': (0.52, 14, [('Labrador Retriever',32,4,.12),('Cocker Spaniel',14,2,.09),('French Bulldog',12,1.5,.09),
                       ('Cockapoo',10,2.5,.11),('Border Collie',18,2.5,.06),('Dachshund',9,1.5,.06),
                       ('German Shepherd',35,5,.06),('Staffordshire Bull Terrier',16,2,.07),('Golden Retriever',30,4,.06),
                       ('Jack Russell Terrier',6.5,1,.07),('Shih Tzu',6,1,.05),('Crossbreed',20,9,.16)]),
    'Cat': (0.38, 16, [('Domestic Shorthair',4.3,.8,.62),('Domestic Longhair',4.6,.9,.10),('British Shorthair',5.5,1,.10),
                       ('Ragdoll',6,1.2,.06),('Bengal',4.8,.8,.06),('Maine Coon',7.5,1.5,.06)]),
    'Rabbit': (0.04, 9, [('Netherland Dwarf',1.0,.15,.3),('Lop',2.2,.4,.5),('Lionhead',1.5,.25,.2)]),
    'Guinea pig': (0.02, 6, [('Abyssinian',1.0,.15,.5),('American',1.0,.15,.5)]),
    'Bird': (0.02, 15, [('Budgerigar',0.035,.005,.6),('Cockatiel',0.09,.01,.4)]),
    'Reptile': (0.02, 15, [('Bearded Dragon',0.45,.08,.6),('Leopard Gecko',0.06,.01,.4)]),
}
# normal body temperature mean/sd by species (reptiles are ectothermic, so not recorded)
TEMP = {'Dog': (38.6, .35), 'Cat': (38.7, .35), 'Rabbit': (39.0, .4), 'Guinea pig': (38.8, .4), 'Bird': (41.0, .4)}

TREATMENTS = [  # code, description, category, 2024 price, sedation
    ('CON-001','Standard consultation','Consultation',52.00,0), ('CON-002','Emergency consultation','Consultation',145.00,0),
    ('CON-003','Nurse clinic','Consultation',28.00,0),          ('CON-004','Post-operative check','Consultation',0.00,0),
    ('VAC-001','Canine booster (DHP + Lepto)','Vaccination',58.00,0), ('VAC-002','Feline booster (FCV, FHV, FPV)','Vaccination',55.00,0),
    ('VAC-003','Kennel cough intranasal','Vaccination',42.00,0), ('VAC-004','Rabbit RHDV2 and myxomatosis','Vaccination',61.00,0),
    ('VAC-005','Puppy or kitten primary course','Vaccination',89.00,0),
    ('DIA-001','In-house blood panel','Diagnostics',118.00,0),   ('DIA-002','Urinalysis','Diagnostics',46.00,0),
    ('DIA-003','Digital radiograph (per view)','Diagnostics',165.00,1), ('DIA-004','Abdominal ultrasound','Diagnostics',210.00,1),
    ('DIA-005','Skin scrape and cytology','Diagnostics',64.00,0),
    ('SUR-001','Bitch spay','Surgery',395.00,1), ('SUR-002','Dog castration','Surgery',310.00,1),
    ('SUR-003','Cat spay','Surgery',145.00,1),   ('SUR-004','Cat castration','Surgery',105.00,1),
    ('SUR-005','Lump removal','Surgery',420.00,1), ('SUR-006','Wound repair','Surgery',285.00,1),
    ('DEN-001','Scale and polish','Dental',320.00,1), ('DEN-002','Dental extraction (per tooth)','Dental',48.00,1),
    ('MED-001','Course of antibiotics','Medication',32.00,0), ('MED-002','NSAID pain relief (per week)','Medication',24.00,0),
    ('MED-003','Prescription diet (bag)','Medication',56.00,0), ('MED-004','Ear drops','Medication',27.00,0),
    ('PAR-001','Flea treatment (monthly dose)','Parasite control',14.50,0), ('PAR-002','Wormer (single dose)','Parasite control',11.00,0),
    ('PAR-003','Tick removal','Parasite control',18.00,0), ('PAR-004','Microchip implant','Parasite control',25.00,0),
]

## 4. Table generators

In [5]:
# --- clinics -------------------------------------------------------------
clinics = pd.DataFrame(CLINICS, columns=['clinic_name','town','county','postcode_district',
                                         'opened_year','consult_rooms','has_24h_emergency'])
clinics.insert(0, 'clinic_id', range(1, len(clinics) + 1))

# --- vets: one clinical director per clinic, then a realistic grade mix --------
GRADE_YEARS = {'Graduate': (2023, 2026), 'Vet': (2013, 2022), 'Senior Vet': (2000, 2014), 'Clinical Director': (1990, 2008)}
vet_rows, used_rcvs, used_mail = [], set(), set()
def make_vet(clinic_id, grade):
    first, last = fake.first_name(), fake.last_name()
    rcvs = str(rng.integers(1_000_000, 9_999_999))
    while rcvs in used_rcvs:
        rcvs = str(rng.integers(1_000_000, 9_999_999))
    used_rcvs.add(rcvs)
    clean_last = ''.join(ch for ch in last.lower() if ch.isalpha())
    mail = f'{first[0].lower()}.{clean_last}@vetcareherts.co.uk'
    n = 2
    while mail in used_mail:
        mail = mail.replace('@', f'{n}@', 1); n += 1
    used_mail.add(mail)
    lo, hi = GRADE_YEARS[grade]
    return [clinic_id, first, last, rcvs, grade, int(rng.integers(lo, hi + 1)), mail]

for cid, rooms in zip(clinics.clinic_id, clinics.consult_rooms):
    vet_rows.append(make_vet(cid, 'Clinical Director'))
    for _ in range(int(rooms) + int(rng.integers(0, 2))):   # bigger clinics employ more vets
        vet_rows.append(make_vet(cid, rng.choice(['Graduate','Vet','Senior Vet'], p=[.25,.5,.25])))
vets = pd.DataFrame(vet_rows, columns=['home_clinic_id','first_name','last_name','rcvs_number','grade','qualified_year','work_email'])
vets.insert(0, 'vet_id', range(1, len(vets) + 1))
print(len(vets), 'vets'); vets.grade.value_counts()

47 vets


grade
Vet                  25
Senior Vet            9
Clinical Director     7
Graduate              6
Name: count, dtype: int64

In [6]:
# --- vet_rota: 3 to 5 days at the home clinic, ~15% also cover one locum day elsewhere ---
rota = set()
rota_rows = []
for v in vets.itertuples():
    days = rng.choice(range(1, 7), size=int(rng.integers(3, 6)), replace=False)  # Mon-Sat
    for d in days:
        rota_rows.append((v.vet_id, v.home_clinic_id, int(d), rng.choice(['Full day','Morning','Afternoon'], p=[.7,.15,.15])))
        rota.add((v.vet_id, v.home_clinic_id, int(d)))
    if rng.random() < .15:
        other = int(rng.choice([c for c in clinics.clinic_id if c != v.home_clinic_id]))
        free = [d for d in range(1, 7) if d not in days]
        if free:
            d = int(rng.choice(free)); rota_rows.append((v.vet_id, other, d, 'Full day')); rota.add((v.vet_id, other, d))
# 24-hour clinics also staff Sundays (weekday 7) with one vet
for cid in clinics.loc[clinics.has_24h_emergency == 1, 'clinic_id']:
    for vid in vets.loc[vets.home_clinic_id == cid, 'vet_id'].sample(2, random_state=SEED):
        if (vid, cid, 7) not in rota:
            rota_rows.append((int(vid), int(cid), 7, 'Full day')); rota.add((int(vid), int(cid), 7))
vet_rota = pd.DataFrame(rota_rows, columns=['vet_id','clinic_id','weekday','shift'])
print(len(vet_rota), 'rota rows')

199 rota rows


In [7]:
# --- owners -----------------------------------------------------------------
N_OWNERS = 1600
MAIL_DOMAINS = (['gmail.com','hotmail.co.uk','outlook.com','yahoo.co.uk','icloud.com','btinternet.com','sky.com'],
                [.38,.17,.14,.11,.09,.07,.04])
clinic_share = (clinics.consult_rooms / clinics.consult_rooms.sum()).values   # bigger clinic, more clients

def email_for(first, last):
    style = rng.integers(0, 4)
    f, l = (''.join(filter(str.isalpha, part.lower())) for part in (first, last))
    local = [f'{f}.{l}', f'{f}{l}{rng.integers(1, 99)}', f'{f[0]}{l}', f'{f}_{l}{rng.integers(70, 99)}'][style]
    return f"{local}@{rng.choice(MAIL_DOMAINS[0], p=MAIL_DOMAINS[1])}"

owner_rows = []
for i in range(N_OWNERS):
    first, last = fake.first_name(), fake.last_name()
    cid = int(rng.choice(clinics.clinic_id, p=clinic_share))
    years_ago = min(rng.exponential(3.2), 11.5)                 # recent registrations are more common
    reg = END - timedelta(days=int(years_ago * 365))
    mobile = f"07{rng.integers(100, 999)} {rng.integers(100000, 999999)}"
    owner_rows.append([first, last, email_for(first, last), mobile, rng.choice(CATCHMENT[cid]), cid,
                       reg.isoformat(), int(rng.random() < .41)])
owners = pd.DataFrame(owner_rows, columns=['first_name','last_name','email','mobile','postcode_district',
                                           'home_clinic_id','registered_on','marketing_consent'])
owners.insert(0, 'owner_id', range(1, len(owners) + 1))
owners.head()

,owner_id,first_name,last_name,email,mobile,postcode_district,home_clinic_id,registered_on,marketing_consent
0,1,John,Rhodes,john_rhodes88@outlook.com,07494 744521,WD25,4,2026-01-19,1
1,2,Abbie,Wallace,abbiewallace28@yahoo.co.uk,07457 980321,LU4,5,2016-07-09,0
2,3,Clare,Roberts,clare_roberts71@outlook.com,07334 586108,AL4,1,2026-07-04,0
3,4,Wendy,Evans,wendy_evans74@gmail.com,07484 182543,HP2,6,2015-04-04,1
4,5,Marc,Smith,marc.smith@icloud.com,07133 893711,SG2,3,2021-03-27,0


In [8]:
# --- pets -------------------------------------------------------------------
sp_names = list(SPECIES); sp_p = [SPECIES[s][0] for s in sp_names]
NAMES = ['Bella','Luna','Milo','Teddy','Daisy','Coco','Max','Poppy','Alfie','Charlie','Bear','Willow','Nala','Simba',
         'Rosie','Ruby','Archie','Oscar','Loki','Tilly','Bonnie','Monty','Pepper','Ziggy','Hazel','Biscuit','Mabel',
         'Otis','Ted','Maple','Rolo','Smudge','Tigger','Misty','Jasper','Honey','Buster','Winnie','Pip','Olive']
pet_rows, chips = [], set()
for o in owners.itertuples():
    n_pets = 1 + rng.poisson(0.45)                                   # most households have one animal
    taken = set()
    for _ in range(n_pets):
        sp = rng.choice(sp_names, p=sp_p)
        _, life, breeds = SPECIES[sp]
        b = breeds[rng.choice(len(breeds), p=np.array([x[3] for x in breeds]) / sum(x[3] for x in breeds))]
        age = float(np.clip(rng.gamma(2.2, life / 6), 0.15, life * 1.2))
        dob = END - timedelta(days=int(age * 365))
        name = rng.choice([n for n in NAMES if n not in taken]); taken.add(name)
        growth = min(1.0, 0.25 + age / 1.0) if age < 1 else 1.0        # puppies and kittens weigh less
        weight = round(float(max(b[1] * growth + rng.normal(0, b[2]), b[1] * .35)), 3 if b[1] < 1 else 1)
        neutered = int(sp in ('Dog','Cat','Rabbit') and age > .5 and rng.random() < (.82 if sp == 'Cat' else .63))
        chip = None
        if sp in ('Dog','Cat') and rng.random() < (.97 if sp == 'Dog' else .78):  # dog chipping is a UK legal duty
            chip = '9' + ''.join(map(str, rng.integers(0, 10, 14)))
            while chip in chips:
                chip = '9' + ''.join(map(str, rng.integers(0, 10, 14)))
            chips.add(chip)
        sex = rng.choice(['Male','Female','Unknown'], p=[.49,.49,.02] if sp in ('Bird','Reptile') else [.5,.5,0])
        pet_rows.append([o.owner_id, name, sp, b[0], sex, dob.isoformat(), weight, neutered, chip])
pets = pd.DataFrame(pet_rows, columns=['owner_id','pet_name','species','breed','sex','date_of_birth',
                                       'weight_kg','neutered','microchip_no'])
pets.insert(0, 'pet_id', range(1, len(pets) + 1))
print(len(pets), 'pets'); pets.species.value_counts()

2351 pets


species
Dog           1203
Cat            885
Rabbit         105
Bird            60
Guinea pig      52
Reptile         46
Name: count, dtype: int64

In [9]:
treatments = pd.DataFrame(TREATMENTS, columns=['treatment_code','description','category','list_price_gbp','requires_sedation'])
tprice = treatments.set_index('treatment_code').list_price_gbp

In [10]:
# --- appointments and appointment_treatments ---------------------------------
REASONS = ['Routine check','Vaccination','Illness','Injury','Dental','Surgery','Follow-up','Emergency']
REASON_P = [.20, .22, .22, .09, .05, .06, .12, .04]
TRIAGE = {'Routine check': [.8,.2,0,0,0], 'Vaccination': [.9,.1,0,0,0], 'Illness': [.1,.35,.35,.15,.05],
          'Injury': [.05,.25,.4,.2,.1], 'Dental': [.3,.5,.2,0,0], 'Surgery': [.1,.4,.35,.15,0],
          'Follow-up': [.6,.35,.05,0,0], 'Emergency': [0,0,.15,.45,.4]}
DURATION = {'Routine check': (15, 3), 'Vaccination': (12, 3), 'Illness': (22, 6), 'Injury': (30, 10),
            'Dental': (75, 20), 'Surgery': (110, 40), 'Follow-up': (12, 3), 'Emergency': (55, 25)}
MONTH_W = np.array([1.15, .9, .95, 1.0, 1.05, 1.1, 1.15, 1.1, 1.0, .95, .9, .75])  # Jan boosters, summer injuries
DAY_W = np.array([1.1, 1.0, 1.0, 1.0, 1.05, 1.2, .25])                              # Mon..Sun
all_days = pd.date_range(START, END)
day_p = MONTH_W[all_days.month - 1] * DAY_W[all_days.day_of_week]
day_p /= day_p.sum()
rota_lookup = vet_rota.groupby(['clinic_id','weekday']).vet_id.apply(list).to_dict()

def choose_treatments(reason, species, triage):
    codes = []
    if reason == 'Emergency': codes.append('CON-002')
    elif reason == 'Follow-up': codes.append('CON-004')
    elif reason == 'Vaccination' and rng.random() < .3: codes.append('CON-003')
    else: codes.append('CON-001')
    if reason == 'Vaccination':
        codes.append({'Dog': 'VAC-001', 'Cat': 'VAC-002', 'Rabbit': 'VAC-004'}.get(species, 'CON-003'))
        if species == 'Dog' and rng.random() < .35: codes.append('VAC-003')
    if reason in ('Illness','Emergency','Injury'):
        meds = ['MED-001','MED-002','MED-004']
        codes += list(rng.choice(meds, size=min(3, 1 + rng.poisson(.3)), replace=False, p=[.45,.4,.15]))
        diag = ['DIA-001','DIA-002','DIA-003','DIA-004','DIA-005']
        k = min(len(diag), rng.binomial(2, .1 * triage))     # sicker animals get more tests
        if k: codes += list(rng.choice(diag, size=k, replace=False, p=[.35,.2,.2,.1,.15]))
    if reason == 'Injury' and rng.random() < .35: codes.append('SUR-006')
    if reason == 'Dental':
        codes.append('DEN-001')
        if rng.random() < .55: codes.append('DEN-002')
    if reason == 'Surgery':
        codes.append(rng.choice(['SUR-001','SUR-002','SUR-003','SUR-004','SUR-005'], p=[.25,.2,.2,.15,.2]))
        codes.append('MED-002')
    if reason == 'Routine check' and rng.random() < .55: codes.append(rng.choice(['PAR-001','PAR-002','PAR-004','MED-003']))
    return list(dict.fromkeys(codes))

pet_info = pets.merge(owners[['owner_id','home_clinic_id']], on='owner_id')
appt_rows, line_rows, booked = [], [], set()
for p in pet_info.itertuples():
    first_day = max(START, date.fromisoformat(p.date_of_birth) + timedelta(days=56))
    n = rng.poisson(5.3 if p.species in ('Dog','Cat') else 2.5)
    for _ in range(n):
        d = all_days[rng.choice(len(all_days), p=day_p)].date()
        if d < first_day: continue
        clinic = p.home_clinic_id if rng.random() < .9 else int(rng.choice(clinics.clinic_id))
        wd = d.isoweekday()
        vets_on = rota_lookup.get((clinic, wd))
        if not vets_on: continue                              # clinic closed that day
        reason = rng.choice(REASONS, p=REASON_P)
        if p.species == 'Reptile' and reason in ('Vaccination','Dental'): reason = 'Routine check'
        triage = int(rng.choice(range(1, 6), p=TRIAGE[reason]))
        slot = int(rng.integers(0, 40))                       # 15-minute slots from 08:30
        start = datetime(d.year, d.month, d.day, 8, 30) + timedelta(minutes=15 * slot)
        if (p.pet_id, start) in booked: continue              # a pet cannot be booked twice at once
        booked.add((p.pet_id, start))
        status = rng.choice(['Completed','Cancelled','No-show'], p=[.89,.06,.05]) if reason != 'Emergency' else 'Completed'
        temp = dur = pay = None
        fee = 0.0
        if status == 'Completed':
            mu, sd = DURATION[reason]
            dur = int(np.clip(round(rng.gamma((mu / sd) ** 2, sd ** 2 / mu)), 5, 480))
            if p.species in TEMP:
                m, s = TEMP[p.species]
                fever = 1.1 if reason in ('Illness','Emergency') and rng.random() < .35 else 0
                temp = round(float(rng.normal(m + fever, s)), 1)
            infl = 1.05 ** ((d - START).days / 365)           # ~5% annual price rise
            for code_ in choose_treatments(reason, p.species, triage):
                qty = int(rng.integers(1, 4)) if code_ in ('DEN-002','DIA-003','MED-002') else 1
                price = round(float(tprice[code_]) * infl * qty, 2)
                line_rows.append([len(appt_rows) + 1, code_, qty, price]); fee += price
            pay = rng.choice(['Card','Cash','Insurance claim','Pet health plan','Bank transfer'], p=[.55,.04,.22,.16,.03])
        appt_rows.append([p.pet_id, int(rng.choice(vets_on)), clinic, start.strftime('%Y-%m-%d %H:%M:%S'),
                          reason, triage, status, temp, dur, round(fee, 2), pay])

appointments = pd.DataFrame(appt_rows, columns=['pet_id','vet_id','clinic_id','start_time','reason','triage_level',
                                                'status','body_temp_c','duration_min','fee_gbp','payment_method'])
appointments.insert(0, 'appointment_id', range(1, len(appointments) + 1))
appt_treat = pd.DataFrame(line_rows, columns=['appointment_id','treatment_code','quantity','line_price_gbp'])
print(len(appointments), 'appointments,', len(appt_treat), 'treatment lines')

10159 appointments, 20574 treatment lines


## 5. Injecting deliberate data-quality issues

Real clinical systems are never perfectly clean, so a realistic amount of missing and duplicate data is
inserted **on purpose**, with fixed rates and logged IDs:

| Issue | Table.column | Rate | Real-world cause it imitates |
|---|---|---|---|
| Missing | owners.mobile | 4% | client declined to give a number |
| Missing | owners.email | 2.5% | older paper registrations |
| Missing | pets.weight_kg | 3% (8% for birds/reptiles) | animal not weighed |
| Missing | appointments.body_temp_c | 3% of completed visits | reading not taken (fractious animal) |
| Missing | appointments.payment_method | 1.5% of completed visits | payment not reconciled yet |
| Duplicate | owners | 1.5% of owners | client re-registered at reception with a typo, different case or extra space |
| Duplicate | appointments | 0.5% of appointments | booking saved twice by double-clicking |


In [11]:
DQ = {}
def blank(df, col, frac, mask=None):
    idx = df.index if mask is None else df.index[mask]
    chosen = rng.choice(idx, size=int(round(len(idx) * frac)), replace=False)
    df.loc[chosen, col] = None
    DQ[f'missing {col}'] = len(chosen)

blank(owners, 'mobile', .04)
blank(owners, 'email', .025)
exotic = pets.species.isin(['Bird','Reptile'])
blank(pets, 'weight_kg', .03, ~exotic); n1 = DQ.pop('missing weight_kg')
blank(pets, 'weight_kg', .08, exotic);  DQ['missing weight_kg'] = n1 + DQ['missing weight_kg']
done = appointments.status.eq('Completed')
blank(appointments, 'body_temp_c', .03, done & appointments.body_temp_c.notna())
blank(appointments, 'payment_method', .015, done)

# duplicate owners: same person registered again later with small formatting differences
src = owners.sample(frac=.015, random_state=SEED)
dups = []
for i, o in enumerate(src.itertuples()):
    variant = i % 3
    first = o.first_name.upper() if variant == 0 else o.first_name + ' ' if variant == 1 else o.first_name
    last = o.last_name if variant != 2 else o.last_name.lower()
    reg = min(date.fromisoformat(o.registered_on) + timedelta(days=int(rng.integers(30, 600))), END)
    dups.append([first, last, o.email.upper() if (o.email and variant == 0) else o.email, o.mobile,
                 o.postcode_district, o.home_clinic_id, reg.isoformat(), o.marketing_consent])
dup_owners = pd.DataFrame(dups, columns=owners.columns[1:])
dup_owners.insert(0, 'owner_id', range(owners.owner_id.max() + 1, owners.owner_id.max() + 1 + len(dup_owners)))
owners = pd.concat([owners, dup_owners], ignore_index=True)
DQ['duplicate owners'] = len(dup_owners)
DQ['duplicate owner source ids'] = src.owner_id.tolist()

# duplicate appointments: identical booking saved twice (new id, no treatment lines of its own)
src_a = appointments.sample(frac=.005, random_state=SEED)
dup_a = src_a.copy()
dup_a['appointment_id'] = range(appointments.appointment_id.max() + 1, appointments.appointment_id.max() + 1 + len(dup_a))
appointments = pd.concat([appointments, dup_a], ignore_index=True)
DQ['duplicate appointments'] = len(dup_a)
DQ['duplicate appointment source ids'] = src_a.appointment_id.tolist()[:10]

for k, v in DQ.items():
    print(f'{k:35s} {v}')

missing mobile                      64
missing email                       40
missing weight_kg                   75
missing body_temp_c                 270
missing payment_method              136
duplicate owners                    24
duplicate owner source ids          [442, 1548, 1045, 827, 1103, 743, 1599, 2, 270, 378, 1491, 282, 1132, 95, 995, 672, 662, 988, 100, 353, 1411, 1168, 956, 237]
duplicate appointments              51
duplicate appointment source ids    [17, 1261, 2534, 4791, 3307, 2234, 5385, 6139, 5724, 6508]


## 6. Load into SQLite (foreign keys enforced)

In [12]:
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
con = sqlite3.connect(DB_PATH)
con.executescript(SCHEMA)
con.execute('PRAGMA foreign_keys = ON')

def load(df, table):
    df = df.astype(object).where(df.notna(), None)
    cols = ', '.join(df.columns); qs = ', '.join('?' * len(df.columns))
    con.executemany(f'INSERT INTO {table} ({cols}) VALUES ({qs})', df.itertuples(index=False, name=None))

for df, t in [(clinics,'clinics'), (vets,'vets'), (vet_rota,'vet_rota'), (owners,'owners'), (pets,'pets'),
              (treatments,'treatments'), (appointments,'appointments'), (appt_treat,'appointment_treatments')]:
    load(df, t)
con.commit()
for t in ['clinics','vets','vet_rota','owners','pets','treatments','appointments','appointment_treatments']:
    print(f"{t:24s}{con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]:>7,}")

clinics                       7
vets                         47
vet_rota                    199
owners                    1,624
pets                      2,351
treatments                   30
appointments             10,210
appointment_treatments   20,574


## 7. Validation checks

In [13]:
print('integrity:', con.execute('PRAGMA integrity_check').fetchone()[0])
print('foreign-key violations:', con.execute('PRAGMA foreign_key_check').fetchall())
def q(sql_text):
    return pd.read_sql_query(sql_text, con)
q('''
SELECT 'owners.mobile' AS col, SUM(mobile IS NULL) AS nulls, COUNT(*) AS rows FROM owners
UNION ALL SELECT 'owners.email', SUM(email IS NULL), COUNT(*) FROM owners
UNION ALL SELECT 'pets.weight_kg', SUM(weight_kg IS NULL), COUNT(*) FROM pets
UNION ALL SELECT 'appointments.body_temp_c (completed, non-reptile)', SUM(a.body_temp_c IS NULL), COUNT(*)
          FROM appointments a JOIN pets p USING(pet_id) WHERE a.status='Completed' AND p.species<>'Reptile'
UNION ALL SELECT 'appointments.payment_method (completed)', SUM(payment_method IS NULL), COUNT(*)
          FROM appointments WHERE status='Completed'
''')

integrity: ok
foreign-key violations: []


,col,nulls,rows
0,owners.mobile,65,1624
1,owners.email,41,1624
2,pets.weight_kg,75,2351
3,"appointments.body_temp_c (completed, non-reptile)",271,9035
4,appointments.payment_method (completed),137,9133


In [14]:
# detecting the injected duplicates with SQL, as an analyst would
q('''
SELECT lower(trim(first_name)) AS first, lower(trim(last_name)) AS last, mobile, COUNT(*) AS copies
FROM owners GROUP BY 1, 2, 3 HAVING COUNT(*) > 1 ORDER BY copies DESC LIMIT 5
''')

,first,last,mobile,copies
0,abbie,wallace,07457 980321,2
1,billy,morris,07422 326662,2
2,carly,cooper,07998 613587,2
3,christian,smith,07264 143034,2
4,damian,brown,07870 784843,2


In [15]:
q('''
SELECT COUNT(*) AS duplicate_bookings FROM (
  SELECT pet_id, vet_id, start_time FROM appointments GROUP BY 1, 2, 3 HAVING COUNT(*) > 1)
''')

,duplicate_bookings
0,51


## 8. Example queries

In [16]:
q('''
SELECT c.town, COUNT(*) AS visits, ROUND(AVG(a.fee_gbp), 2) AS avg_fee_gbp,
       ROUND(100.0 * SUM(a.status = 'No-show') / COUNT(*), 1) AS no_show_pct
FROM appointments a JOIN clinics c USING (clinic_id)
GROUP BY c.town ORDER BY visits DESC
''')

,town,visits,avg_fee_gbp,no_show_pct
0,Stevenage,2114,148.29,5.1
1,St Albans,1998,153.92,4.8
2,Luton,1374,159.67,4.6
3,Hatfield,1301,152.30,4.8
4,Watford,1283,156.41,4.3
5,Welwyn Garden City,1227,160.70,5.0
6,Hemel Hempstead,913,158.04,4.8


In [17]:
q('''
SELECT p.species, COUNT(DISTINCT p.pet_id) AS pets, ROUND(AVG(p.weight_kg), 2) AS mean_kg,
       ROUND(AVG(a.body_temp_c), 2) AS mean_temp_c
FROM pets p LEFT JOIN appointments a USING (pet_id)
GROUP BY p.species ORDER BY pets DESC
''')

,species,pets,mean_kg,mean_temp_c
0,Dog,1203,17.76,38.71
1,Cat,885,4.77,38.80
2,Rabbit,105,1.77,39.04
3,Bird,60,0.05,41.14
4,Guinea pig,52,1.05,38.88
5,Reptile,46,0.25,NaN


In [18]:
q('''
SELECT t.category, SUM(at.quantity) AS units, ROUND(SUM(at.line_price_gbp), 0) AS revenue_gbp
FROM appointment_treatments at JOIN treatments t USING (treatment_code)
GROUP BY t.category ORDER BY revenue_gbp DESC
''')

,category,units,revenue_gbp
0,Consultation,9117,474302.0
1,Diagnostics,2309,301472.0
2,Surgery,839,263127.0
3,Medication,7068,208247.0
4,Dental,912,176418.0
5,Vaccination,2308,135170.0
6,Parasite control,775,14001.0


In [19]:
con.close()
print('Saved', DB_PATH, round(os.path.getsize(DB_PATH) / 1e6, 2), 'MB')

Saved vetcare_clinics.db 2.8 MB
